In [2]:
!python3 -m pip install requests


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [3]:
import requests
import json
import os
import re

/Users/user/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
DATA_DIR = "data/literature"
os.makedirs(DATA_DIR, exist_ok=True)

In [5]:
def build_query(gene, variant):
    if variant:
        return f"({gene} AND {variant}) AND (mutation OR disease)"
    return gene

In [6]:
def clean_text(text):
    if not text:
        return ""
    
    text = re.sub(r"<.*?>", "", text)   # remove HTML tags
    text = re.sub(r"\s+", " ", text)    # remove extra spaces
    
    return text.strip()

In [7]:
def fetch_europepmc(query, page_size=50):
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    
    params = {
        "query": query,
        "format": "json",
        "pageSize": page_size,
        "resultType": "core"
    }

    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        
        papers = []
        for item in data.get("resultList", {}).get("result", []):
            papers.append({
                "pmid": item.get("pmid"),
                "title": item.get("title"),
                "abstract": clean_text(item.get("abstractText", ""))
            })
        
        return papers
    
    except Exception as e:
        print("❌ Europe PMC Error:", e)
        return []

In [ ]:
NCBI_API_KEY = os.getenv("NCBI_API_KEY", "")

def fetch_pubmed_fallback(query):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": 50
    }
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    
    try:
        resp = requests.get(url, params=params)
        data = resp.json()
        
        ids = data.get("esearchresult", {}).get("idlist", [])
        
        papers = []
        for pmid in ids:
            papers.append({
                "pmid": pmid,
                "title": "",
                "abstract": ""
            })
        
        return papers
    
    except Exception as e:
        print("❌ PubMed Error:", e)
        return []

In [9]:
def save_corpus(gene, variant, papers):
    filename = f"{gene}_{variant}_corpus.json"
    path = os.path.join(DATA_DIR, filename)
    
    with open(path, "w") as f:
        json.dump(papers, f, indent=4)
    
    print(f"✅ Saved to {path}")

In [10]:
def fetch_literature(gene, variant):
    query = build_query(gene, variant)
    
    print(f"🔍 Query: {query}")
    
    papers = fetch_europepmc(query)
    
    if not papers:
        print("⚠️ Europe PMC failed → switching to PubMed...")
        papers = fetch_pubmed_fallback(query)
    
    print(f"📄 Papers fetched: {len(papers)}")
    
    save_corpus(gene, variant, papers)
    
    return papers

In [11]:
papers = fetch_literature("SNCA", "A53T")

papers[:3]  # preview first 3

🔍 Query: (SNCA AND A53T) AND (mutation OR disease)
📄 Papers fetched: 50
✅ Saved to data/literature/SNCA_A53T_corpus.json


[{'pmid': None,
  'title': "Characterization of ATP10B R303W as a Novel Pathogenic Mutation in Parkinson's Disease",
  'abstract': ''},
 {'pmid': None,
  'title': 'The <i>SNCA</i> A53T mutation sensitizes human neurons and microglia to ferroptosis',
  'abstract': 'The major pathological hallmarks of sporadic and familial forms of Parkinson’s disease (PD) are the targeted and progressive loss of midbrain dopaminergic neurons (mDA), associated with systemic iron accumulation, α-synuclein (αsyn) accumulation and aggregation, and lipid peroxidation amongst other reactive oxygen species (ROS) generation. Therapeutic strategies aimed towards dopamine restoration, αsyn removal and iron chelation have provided symptomatic relief but failed to prevent or slow disease progression. This is in part due to the lack of understanding of the exact pathways leading to neuronal death in PD. In this study, we investigate ferroptosis, a unique cell death mechanism sharing multiple features with PD patholo

In [14]:
papers[7]

{'pmid': '41721127',
 'title': 'Alpha-synuclein overexpression triggers divergent cellular responses and post-translational modifications in SH-SY5Y and ReNcell VM models.',
 'abstract': "Alpha‑synuclein (α‑syn) overexpression models are widely used to unravel the molecular mechanisms of Parkinson's disease (PD), particularly in light of the dose-dependent transition between its physiological and toxic roles. However, existing systems rely on inducible expression, lack robust dose stratification and comparative cellular contexts. Here, we developed and characterized a panel of stable neuronal cell lines in two human cellular models (SH‑SY5Y neuroblastoma cells and ReNcell VM neural progenitors) overexpressing GFP-tagged wild-type (WT) or A53T mutant α‑syn at low and high overexpression levels. Utilizing this framework, we demonstrated that A53T consistently induces cytotoxicity, oxidative stress and mitochondrial dysfunction in both cell types. In contrast, WT α‑syn had divergent effec

In [18]:
papers[4]

{'pmid': None,
 'title': 'Increased serum Neurofilament light chain levels in Parkinson’s Disease patients carrying the p.A53T <i>SNCA</i> mutation: Data from the PPMI study',
 'abstract': 'ABSTRACT Introduction Serum neurofilament light chain (NfL) levels, a marker of axonal damage, are generally elevated in neurodegenerative conditions, but results in idiopathic Parkinson’s Disease (iPD) have been inconsistent. The p.A53T mutation in the SNCA gene usually leads to a severe, rapidly progressing genetic form of PD. Objectives Assessing the possible use of serum NfL as a marker of ongoind neurodegeneration in A53T-PD. Methods Using a propensity score matching based technique with data from the Parkinson’s Progression Markers Initiative, A53T-PD participants were matched, in a 3:1 ratio, to iPD patients and healthy controls by age and disease duration, where applicable. Results After matching, 18 A53T-PD, 54 iPD, and 54 controls were analyzed. Serum NfL was significantly higher in A53T-P